# Two-Stage Offline–Online SOH Prediction Framework
Implements:
- Offline: clustering -> state mapping g(theta)->z -> per-state SOH model
- Online: feature extraction -> infer state -> predict SOH -> change-point detection for switching




# Imports

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataclasses import dataclass

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA


from scipy.spatial.distance import mahalanobis
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.linalg import pinvh

# Global Configs

In [ ]:
# --- Random seed for reproducibility --- #
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# --- Output Dir --- #
ARTIFACT_DIR = "outputs"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# --- BOL and EOL SOH Value Threshold --- #
E_BOL = 0.90
E_EOL = 0.75
# For BOL-EOL guided clustering merging threshold gamma (tune on validation)
GAMMA_MERGE = 0.40
# Default number of initial bins for BOL-EOL guided clustering
K_INIT_BINS = 8

# Fixed number of health states K_final, set it; else None
K_FINAL = 10  # e.g., 5

# Offline Stage

## Load Offline Data

In [ ]:
def load_offline_dataset(file_path, feature_cols, soh_col='SOH', id_cols=['CELL','SOH_label','SOC_label']):
    if feature_cols is None or not isinstance(feature_cols, (list, tuple)) or len(feature_cols) == 0:
        raise ValueError("feature_cols must be a non-empty list of column names (strings).")
    # Ensure all are strings
    feature_cols = [str(c) for c in feature_cols]

    df = pd.read_csv(file_path)
    # NOTE: HARD CODE to REMOVE THE OUTLIERS
    df = df[~((df["CELL"] == "CELL050") & (df["SOH_label"] == 1) & (df["SOC_label"] == 5))]


    if soh_col not in df.columns:
        raise ValueError(f"Missing label column '{soh_col}' in file.")
    #NOTE: change SOH to range [0,1] if needed
    df[soh_col] = df[soh_col].astype(float) / 3.600
    
    missing_id_cols = [c for c in id_cols if c not in df.columns]
    if missing_id_cols:
        raise ValueError(f"Missing id columns in file: {missing_id_cols}")
    # Create Coposite ID
    new_id_col = "ID"
    df[new_id_col] = (
        df[id_cols]
        .apply(
            lambda col: col.astype(int).astype(str) if col.name in ["SOH_label", "SOC_label"] else col.astype(str)
        )
        .agg("_".join, axis=1)
    )

    missing = [c for c in feature_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing feature columns in file: {missing}")

    required_cols = list(id_cols) + [soh_col] + list(feature_cols)

    # Handle NaNs
    if df[required_cols].isna().any().any():
        before = len(df)
        df = df.dropna(subset=required_cols).reset_index(drop=True)
        after = len(df)
        print(f"Dropped {before - after} rows with NaNs in required columns.")
       

    # Convert to numeric safely
    X_df = df[feature_cols].apply(pd.to_numeric, errors="coerce")
    y_s = pd.to_numeric(df[soh_col], errors="coerce")

    # After coercion, check again
    if X_df.isna().any().any() or y_s.isna().any():
        bad_cols = X_df.columns[X_df.isna().any()].tolist()
        bad_y = bool(y_s.isna().any())
        msg = []
        if bad_cols:
            msg.append(f"Non-numeric values (coerced to NaN) in feature columns: {bad_cols}")
        if bad_y:
            msg.append(f"Non-numeric values (coerced to NaN) in '{soh_col}'")
        
        before = len(df)
        keep = (~X_df.isna().any(axis=1)) & (~y_s.isna())
        df = df.loc[keep].reset_index(drop=True)
        X_df = X_df.loc[keep].reset_index(drop=True)
        y_s = y_s.loc[keep].reset_index(drop=True)
        after = len(df)
        if before != after:
            print(f"Dropped {before - after} rows with NaNs in feature or label columns.")


    X = X_df.values.astype(float)
    y = y_s.values.astype(float)

    meta = {
        "file_path": file_path,
        "n_rows": int(len(df)),
        "n_features": int(X.shape[1]),
        "feature_cols": list(feature_cols),
        "soh_col": soh_col,
        "id_cols": list(id_cols),
        "new_id_col": new_id_col,
    }
    return df, X, y, meta

## Clustering [Method 1]: Hierarchical Clustering

**Clustering Option 1: Hierarchical Clustering (Ward, Euclidean)**
 
Produces hard assignment z in {1..K} and a helper to assign new samples.


In [ ]:
@dataclass
class HierarchicalStateModel:
    scaler: StandardScaler
    centroids: np.ndarray  # shape (K, d)
    K: int
    Z: np.ndarray  # linkage matrix (for inspection / dendrogram)

    def predict(self, X):
        Xs = self.scaler.transform(X)
        dists = pairwise_distances(Xs, self.centroids, metric="euclidean")
        z = np.argmin(dists, axis=1) + 1 # 1-based cluster labels
        #NOTE: soft-ish membership via distance -> similarity (not a true probability)
        sim = np.exp(-dists)
        p = sim / (sim.sum(axis=1, keepdims=True) + 1e-12)
        return z, p


def fit_hierarchical_states(X, K=5, metric="euclidean", y=None):
    """
    SciPy Ward hierarchical clustering.

    Parameters
    ----------
    X : array (n, d)
        Feature matrix (already your θ / ECM params).
    K : int
        Number of clusters (health states).
    metric : str
        Distance metric for pdist (Ward linkage expects Euclidean in theory).
        Keep 'euclidean' unless you have a strong reason.

    Returns
    -------
    state_model : HierarchicalStateModel
    z : array (n,) labels in {1..K}
    """
    if K < 2:
        raise ValueError("K must be >= 2")

    scaler = StandardScaler()
    Xs = scaler.fit_transform(np.asarray(X, dtype=float))

    # Condensed pairwise distances then Ward linkage
    # NOTE: Ward is defined for Euclidean distances; keep metric="euclidean".
    dcond = pdist(Xs, metric=metric)
    Z = linkage(dcond, method="ward")

    # Cut the dendrogram to get K clusters.
    # fcluster returns labels starting at 1.
    z = fcluster(Z, t=K, criterion="maxclust")
    
    # If y is not none, reordered by max SOH value
    if y is not None:
        state_soh = []
        for k in range(1, K + 1):
            idx = (z == k)
            if idx.sum() == 0:
                state_soh.append(-np.inf)  # empty cluster
            else:
                state_soh.append(y[idx].max())
        # Get order of states by descending max SOH
        order = np.argsort(state_soh)[::-1]  # indices of states in new order
        # Create mapping from old state to new state
        old_to_new = {old_k + 1: new_k + 1 for new_k, old_k in enumerate(order)}
        # Reassign z
        z = np.array([old_to_new[old_k] for old_k in z])
    


    # Compute centroids in scaled space for prediction
    centroids = []
    for k in range(1, K + 1):
        idx = (z == k) # boolean index (samples in cluster k)
        if idx.sum() == 0:
            # Rare case fallback to global mean
            centroids.append(Xs.mean(axis=0))
        else:
            centroids.append(Xs[idx].mean(axis=0))
    centroids = np.vstack(centroids)

    model = HierarchicalStateModel(
        scaler=scaler,
        centroids=centroids,
        K=K,
        Z=Z
    )
    return model, z


def plot_dendrogram(Z, labels=None, truncate_mode=None, p=30, title="Hierarchical clustering dendrogram"):
    """
    Z: linkage matrix from scipy.cluster.hierarchy.linkage
    labels: optional array of labels for samples (default None)
    truncate_mode: None, 'lastp', 'level'
    p: parameter used with truncate_mode
    """
    plt.figure(figsize=(10, 4))
    if labels is not None:
        dendrogram(Z, labels=labels, truncate_mode=truncate_mode, p=p, leaf_rotation=90, leaf_font_size=8)
    else:
        dendrogram(Z, truncate_mode=truncate_mode, p=p, no_labels=True)
    plt.title(title)
    plt.xlabel("samples")
    plt.ylabel("merge distance")
    plt.tight_layout()
    plt.show()

## Clustering [Method 2]: BOL & EOL Guided

**Clustering Option 2: BOL–EOL Guided Clustering**


Steps (per your PDF):
1) Define BOL and EOL sets using SOH thresholds
2) Compute Mahalanobis ratio h(θ)
3) Sort by h and split into K_init bins (with BOL/EOL anchored)
4) Merge adjacent bins if separability J(A,B) < gamma


Returns:
- a "binning + merge" assignment for offline data
- a nearest-centroid predictor for online assignment

Practical note:
- Mahalanobis needs covariance inverses; we use pseudo-inverse with pinvh.

In [ ]:
@dataclass
class BOLEOLStateModel:
    scaler: StandardScaler
    centroids: np.ndarray
    K: int
    meta: dict

    def predict(self, X):
        Xs = self.scaler.transform(X)
        dists = pairwise_distances(Xs, self.centroids, metric="euclidean")
        z = np.argmin(dists, axis=1) + 1
        #NOTE: soft-ish membership via distance -> similarity (not a true probability)
        sim = np.exp(-dists)
        p = sim / (sim.sum(axis=1, keepdims=True) + 1e-12)
        return z, p


def _mahalanobis_distance_batch(X, mu, cov):
    cov_inv = pinvh(cov) # alter: cov_inv = np.linalg.inv(cov)
    Xm = X - mu
    # d^2 = (x-mu)T cov_inv (x-mu)
    d2 = np.einsum("ij,jk,ik->i", Xm, cov_inv, Xm)
    d2 = np.maximum(d2, 0.0)
    return np.sqrt(d2)


def fit_boleol_guided_states(X, y, e_bol=0.90, e_eol=0.75, k_init=8, gamma=0.40, k_final=None):
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    # Define BOL/EOL sets
    bol_mask = y >= e_bol
    eol_mask = y <= e_eol
    if bol_mask.sum() < 5 or eol_mask.sum() < 5:
        print(f"BOL samples: {bol_mask.sum()}, EOL samples: {eol_mask.sum()}")
        raise ValueError("Not enough samples (<5) in BOL or EOL sets. Adjust e_bol/e_eol or check data.")

    mu_bol = Xs[bol_mask].mean(axis=0)
    mu_eol = Xs[eol_mask].mean(axis=0)
    cov_bol = np.cov(Xs[bol_mask].T)
    cov_eol = np.cov(Xs[eol_mask].T)

    d_bol = _mahalanobis_distance_batch(Xs, mu_bol, cov_bol)
    d_eol = _mahalanobis_distance_batch(Xs, mu_eol, cov_eol)
    h = d_bol / (d_bol + d_eol + 1e-12)  # in [0,1] typically
    print(f"Min h: {h.min()}, Max h: {h.max()}")
    print(h)

    order = np.argsort(h)
    Xs_sorted = Xs[order]
    # initial bins
    bins = np.array_split(np.arange(len(Xs_sorted)), k_init)
    print(len(bins), "initial bins created.")

    # Compute adjacent separability score J(A,B)
    def cluster_stats(idxs):
        Xa = Xs_sorted[idxs]
        mu = Xa.mean(axis=0)
        cov = np.cov(Xa.T) if Xa.shape[0] > 1 else np.eye(Xa.shape[1])
        return mu, cov, Xa

    def J(idxsA, idxsB):
        muA, covA, XA = cluster_stats(idxsA)
        muB, covB, XB = cluster_stats(idxsB)
        num = np.linalg.norm(muA - muB)
        den = np.trace(covA) + np.trace(covB) + 1e-12
        return num / np.sqrt(den)

    # Merge adjacent bins with smallest separability until all >= gamma or reach k_final
    bins_list = [b.copy() for b in bins]

    while True:
        if len(bins_list) <= 2:
            break
        if k_final is not None and len(bins_list) <= k_final:
            break

        scores = []
        for i in range(len(bins_list) - 1):
            scores.append(J(bins_list[i], bins_list[i+1]))
        scores = np.array(scores)

        min_idx = int(np.argmin(scores))
        min_score = float(scores[min_idx])

        if min_score >= gamma:
            break

        # merge bins min_idx and min_idx+1
        merged = np.concatenate([bins_list[min_idx], bins_list[min_idx+1]])
        bins_list = bins_list[:min_idx] + [merged] + bins_list[min_idx+2:]

    # Assign labels in sorted space
    labels_sorted = np.zeros(len(Xs_sorted), dtype=int)
    for k, idxs in enumerate(bins_list):
        labels_sorted[idxs] = k + 1

    # map back to original order
    labels = np.zeros(len(Xs), dtype=int)
    labels[order] = labels_sorted

    # centroids
    K = len(bins_list)
    centroids = np.vstack([Xs[labels == (k+1)].mean(axis=0) for k in range(K)])
    print(f"Fitted BOL-EOL guided clustering with K={K} states (final).")

    meta = {
        "e_bol": e_bol,
        "e_eol": e_eol,
        "k_init": k_init,
        "gamma": gamma,
        "k_final": k_final,
        "K_learned": K
    }
    return BOLEOLStateModel(scaler=scaler, centroids=centroids, K=K, meta=meta), labels


## Per-State SOH Estimator E(y|z)

From health state to SOH Value, we have different mode:
- "mean": constant mean SOH per state (very robust, low variance)
- "ridge": linear ridge regression per state

We define a generic trainer and predictor using either "hierarchical" or "boleol" clustering

In [ ]:
@dataclass
class StateSOHEstimator:
    mode: str
    models: dict          # state -> model or float mean
    feature_cols: list
    state_model_kind: str # "hierarchical" or "boleol"
    state_model: object

    def predict(self, X):
        z, p = self.state_model.predict(X)
        yhat = np.zeros(X.shape[0], dtype=float)

        if self.mode == "mean":
            for i in range(X.shape[0]):
                yhat[i] = float(self.models[int(z[i])])
            return yhat, z, p

        # regression modes
        for i in range(X.shape[0]):
            zi = int(z[i])
            mdl = self.models.get(zi)
            if mdl is None:
                # fallback: nearest state's mean if missing
                yhat[i] = np.nan
            else:
                yhat[i] = float(mdl.predict(X[i:i+1])[0])

        # fill missing with global mean if any
        if np.isnan(yhat).any():
            global_mean = np.mean([v for v in self.models.values() if isinstance(v, (int, float, np.floating))] or [0.85])
            yhat = np.where(np.isnan(yhat), global_mean, yhat)

        return yhat, z, p


def train_state_soh_estimator(X, y, z, feature_cols, state_model, state_model_kind, mode="mean"):
    models = {}
    for k in sorted(np.unique(z)):
        idx = (z == k)
        Xk, yk = X[idx], y[idx]

        if mode == "mean":
            models[int(k)] = float(np.mean(yk))
            continue

        if len(yk) < 20:
            # too few samples -> fallback to mean
            models[int(k)] = float(np.mean(yk))
            continue

        if mode == "ridge":
            mdl = Pipeline([
                ("scaler", StandardScaler()),
                ("reg", Ridge(alpha=1.0, random_state=RANDOM_SEED))
            ])
            mdl.fit(Xk, yk)
            models[int(k)] = mdl

        # elif mode == "rf":
        #     mdl = RandomForestRegressor(
        #         n_estimators=300,
        #         random_state=RANDOM_SEED,
        #         min_samples_leaf=3,
        #         n_jobs=-1
        #     )
        #     mdl.fit(Xk, yk)
        #     models[int(k)] = mdl

        else:
            raise ValueError("mode must be one of: 'mean', 'ridge'")

    return StateSOHEstimator(
        mode=mode,
        models=models,
        feature_cols=feature_cols,
        state_model_kind=state_model_kind,
        state_model=state_model
    )

# Offline Training Pipeline

Choose clustering method:
- method="hierarchical"
- method="boleol"


Choose SOH estimator mode:
- "mean", "ridge"

Visualization Functions

In [ ]:
#1) PCA 2D scatter: colored by cluster; optionally overlay SOH


def plot_clusters_pca2d(X, z, y=None, title="Clusters in PCA space"):
    X = np.asarray(X, dtype=float)
    z = np.asarray(z)

    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    pca = PCA(n_components=2, random_state=0)
    X2 = pca.fit_transform(Xs)
    print(f"PCA Eigenvectors: {pca.components_}")

    # marker cycle (extend if K is large)
    markers = ['o', 's', '^', 'D', 'v', 'P', 'X', '*', '<', '>']

    plt.figure()

    if y is None:
        # cluster only
        for i, k in enumerate(sorted(np.unique(z))):
            idx = (z == k)
            plt.scatter(
                X2[idx, 0], X2[idx, 1],
                marker=markers[i % len(markers)],
                s=18,
                label=f"Cluster {k}"
            )
        plt.legend()

    else:
        # SOH as color, cluster as marker
        for i, k in enumerate(sorted(np.unique(z))):
            idx = (z == k)
            sc = plt.scatter(
                X2[idx, 0], X2[idx, 1],
                c=y[idx],
                marker=markers[i % len(markers)],
                s=18,
                label=f"Cluster {k}",
                vmin=0.7,
                vmax=1.0
            )

            # annotate centroid
            cx, cy = X2[idx].mean(axis=0)
            plt.text(cx, cy, str(int(k)), fontsize=12, weight="bold")

        plt.colorbar(sc, label="SOH")
        # plt.legend(title="Cluster")

    plt.title(title)
    plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    plt.grid(True, alpha=0.2)
    plt.show()

In [ ]:
def plot_soh_by_bin(z, y, title="SOH distribution by BOL-EOL bins", kind="box"):
    """
    z: (n,) integer labels in {1..K} from fit_boleol_guided_states
    y: (n,) SOH values
    kind: "box" or "violin"
    """
    z = np.asarray(z)
    y = np.asarray(y, dtype=float)

    bins = sorted(np.unique(z))
    data = [y[z == k] for k in bins]
    counts = [len(v) for v in data]
    labels = [f"{int(k)} (n={c})" for k, c in zip(bins, counts)]

    plt.figure(figsize=(max(6, 0.9*len(bins)), 4))
    if kind == "violin":
        plt.violinplot(data, showmeans=True, showextrema=True)
        plt.xticks(range(1, len(labels)+1), labels, rotation=0)
    else:
        plt.boxplot(data, labels=labels, showfliers=True)

    plt.xlabel("Group")
    plt.ylabel("SOH")
    plt.title(title)
    plt.grid(True, axis="y", alpha=0.2)
    plt.tight_layout()
    plt.show()

Print Info:

offline trainer:

In [ ]:
def offline_train(
    offline_csv,
    feature_cols,
    id_cols,
    clustering_method="boleol",
    K_hier=5,
    soh_estimator_mode="mean",
    e_bol=E_BOL,
    e_eol=E_EOL,
    k_init_bins=K_INIT_BINS,
    gamma_merge=GAMMA_MERGE,
    k_final=K_FINAL,
):
    df, X, y, meta = load_offline_dataset(offline_csv, feature_cols=feature_cols, id_cols=id_cols)
    print(f"Loaded offline dataset from {offline_csv}: {X.shape[0]} samples, {X.shape[1]} features.")
   
    if clustering_method == "hierarchical":
        state_model, z = fit_hierarchical_states(X, K=K_hier, metric="euclidean", y=y)
        state_model_kind = "hierarchical"

        # plot_dendrogram(state_model.Z, labels=None, truncate_mode="lastp", p=30, title="Hierarchical Clustering Dendrogram")
        plot_dendrogram(state_model.Z, labels=df["ID"].astype(str).tolist(), truncate_mode=None, p=30, title="Hierarchical Clustering Dendrogram")
        combined_labels = [f"{id_}-SOH:{soh:.4f}" for id_, soh in zip(df["CELL"].astype(str), df["SOH"].round(4))]
        plot_dendrogram(state_model.Z, labels=combined_labels, truncate_mode=None, p=30, title="Hierarchical Clustering Dendrogram")

        plot_clusters_pca2d(X, z, y=y, title="Offline clusters (PCA) with SOH colormap\n Hierarchical")
        plot_soh_by_bin(z, y, kind="violin", title="SOH distribution by hierarchical clusters")
        
        # Print State membership summary
        for k in sorted(np.unique(z)):
            idx = (z == k)
            print(f"Group {k}: {idx.sum()} samples, SOH range [{y[idx].min():.4f}, {y[idx].max():.4f}]")
            print("  IDs:", df.loc[idx, "ID"].astype(str).tolist())
            # combine  id, soh and soc
            df_subset = df.loc[idx, ["CELL", "SOH", "SOC"]]
            for _, row in df_subset.iterrows():
                print(f"{row['CELL']}: SOH={row['SOH']:.4f}, SOC={row['SOC']:.4f}")
            
        # Print Distance between state centroids
        centroids = state_model.centroids
        dists = pairwise_distances(centroids, metric="euclidean")
        print("Inter-state centroid distances (euclidean):")
        # readale print
        for i in range(dists.shape[0]):
            row = "  ".join([f"{dists[i,j]:.4f}" for j in range(dists.shape[1])])
            print(row)

       # Individual sample pairwise distances (for analysis)
        sample_dists = pairwise_distances(X, metric="euclidean")
        labels = [f"G{g}-{cell}-SOH:{soh:.4f}-SOC:{soc:.4f}" for g, cell, soh, soc in zip(z, df["CELL"].astype(str), df["SOH"], df["SOC"])]
        sample_dist_df = pd.DataFrame(sample_dists, index=labels, columns=labels)
        out_path = os.path.join(ARTIFACT_DIR, "sample_pairwise_distances_euclidean.csv")
        sample_dist_df.to_csv(out_path)
        print("Saved:", out_path)



    elif clustering_method == "boleol":
        state_model, z = fit_boleol_guided_states(
            X, y, e_bol=e_bol, e_eol=e_eol, k_init=k_init_bins, gamma=gamma_merge, k_final=k_final
        )
        state_model_kind = "boleol"

        plot_clusters_pca2d(X, z, y=y, title="Offline clusters (PCA) with SOH colormap\n BOL-EOL guided")
        plot_soh_by_bin(z, y, kind="violin")

    else:
        raise ValueError("clustering_method must be 'hierarchical' or 'boleol'")

    # Fit per-state SOH estimator
    estimator = train_state_soh_estimator(
        X=X, y=y, z=z,
        feature_cols=feature_cols,
        state_model=state_model,
        state_model_kind=state_model_kind,
        mode=soh_estimator_mode
    )

    # Quick offline evaluation split (sanity check) ? Is it valid
    Xtr, Xte, ytr, yte, ztr, zte = train_test_split(X, y, z, test_size=0.2, random_state=RANDOM_SEED)
    yhat_te, zhat_te, _ = estimator.predict(Xte)
    mae = np.mean(np.abs(yhat_te - yte))
    rmse = np.sqrt(np.mean((yhat_te - yte) ** 2))

    # Save artifacts
    artifact = {
        "clustering_method": clustering_method,
        "soh_estimator_mode": soh_estimator_mode,
        "feature_cols": feature_cols,
        "id_cols": id_cols,
        "metrics": {"mae": float(mae), "rmse": float(rmse)},
    }
    joblib.dump(estimator, os.path.join(ARTIFACT_DIR, f"{clustering_method}_state_soh_estimator.joblib"))
    with open(os.path.join(ARTIFACT_DIR, f"{clustering_method}_meta.json"), "w") as f:
        json.dump(artifact, f, indent=2)

    # Also attach labels to df for inspection
    out_df = df.copy()
    out_df["health_state_z"] = z

    return estimator, out_df, artifact

In [ ]:
# df, X, y, meta = load_offline_dataset (
#     file_path="../fulldf_date_removeAbOod_G40L80SOC_best_temp25.csv",
#     feature_cols=['R0','R1','R2','R3','C1','n1','C2','n2', 'C3','n3','Aw'],
#     id_cols=['CELL','SOH_label','SOC_label'])

# # df.head()
# print(df["SOH"].min(), df["SOH"].max())

In [ ]:
estimator, out_df, artifact = offline_train(
    offline_csv="../fulldf_date_removeAbOod_G40L80SOC_best_temp25.csv",
    feature_cols=['R0','R1','R2','R3','C1','n1','C2','n2', 'C3','n3','Aw'],
    id_cols=['CELL','SOH_label','SOC_label'],
    clustering_method="hierarchical",
    K_hier=8,
    soh_estimator_mode="mean",
)


In [ ]:
# Centroid distance matrix provided earlier
dists = np.array([
    [0.0000, 4.4583, 6.4122, 5.8737, 4.1542, 7.1429, 7.3219, 7.7709],
    [4.4583, 0.0000, 3.7899, 3.8523, 2.1979, 4.1255, 4.7201, 6.1975],
    [6.4122, 3.7899, 0.0000, 4.6823, 4.4923, 4.6081, 5.6242, 6.1230],
    [5.8737, 3.8523, 4.6823, 0.0000, 4.9084, 4.3543, 4.0621, 4.4682],
    [4.1542, 2.1979, 4.4923, 4.9084, 0.0000, 4.5207, 5.0531, 6.2651],
    [7.1429, 4.1255, 4.6081, 4.3543, 4.5207, 0.0000, 1.7699, 4.1925],
    [7.3219, 4.7201, 5.6242, 4.0621, 5.0531, 1.7699, 0.0000, 3.1848],
    [7.7709, 6.1975, 6.1230, 4.4682, 6.2651, 4.1925, 3.1848, 0.0000]
])

# Sequential distances: G1-G2, G2-G3, ..., G7-G8
seq_dists = [dists[i, i+1] for i in range(7)]
groups = [f"G{i}->G{i+1}" for i in range(1, 8)]

# groups = ["G1\n CELL42", "G2\n CELL42 & CELL50", "G3\n CELL45", "G4\n CELL76", "G5\n CELL13 & CELL54", "G6\n CELL50", "G7\n CELL42", "G8\n CELL42"]

plt.figure()
plt.plot(groups, seq_dists, marker='o')
plt.xlabel("Groups")
plt.ylabel("Euclidean distance between consecutive centroids")
plt.title("Centroids Euclidean Distances from G1 to G8")
plt.show()

In [ ]:
import re
import pandas as pd

# If you already have df_dist as your 65x65 matrix, skip loading.
# Otherwise load the CSV you saved:
df_dist = pd.read_csv(os.path.join(ARTIFACT_DIR, "sample_pairwise_distances_euclidean.csv"), index_col=0)

# Parse SOH and SOC from labels
def parse_soh_soc(label: str):
    # Supports decimals and scientific notation just in case
    m_soh = re.search(r"SOH:([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)", label)
    m_soc = re.search(r"SOC:([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)", label)
    soh = float(m_soh.group(1)) if m_soh else float("nan")
    soc = float(m_soc.group(1)) if m_soc else float("nan")
    return soh, soc

meta = pd.DataFrame(index=df_dist.index)
meta[["SOH", "SOC"]] = meta.index.to_series().apply(lambda s: pd.Series(parse_soh_soc(s)))

# Sort: SOH descending, then SOC descending
meta_sorted = meta.sort_values(["SOH", "SOC"], ascending=[False, False])

# Reorder the distance matrix
df_sorted = df_dist.loc[meta_sorted.index, meta_sorted.index]

# Draw horizontal/vertical lines where SOH changes (or SOC changes)
soh_vals = meta_sorted["SOH"].values
soc_vals = meta_sorted["SOC"].values

soh_change_idx = [i for i in range(1, len(soh_vals)) if soh_vals[i] != soh_vals[i-1]]
soc_change_idx = [i for i in range(1, len(soc_vals)) if (soh_vals[i] == soh_vals[i-1]) and (soc_vals[i] != soc_vals[i-1])]

plt.figure(figsize=(10, 8))
plt.imshow(df_sorted.values, aspect="auto")
plt.colorbar(label="Euclidean distance")
plt.title("Pairwise distances with SOH/SOC ordering")
plt.xlabel("Samples (SOH↓, SOC↓)")
plt.ylabel("Samples (SOH↓, SOC↓)")

for i in soh_change_idx:
    plt.axhline(i - 0.5, linewidth=1)
    plt.axvline(i - 0.5, linewidth=1)

# Optional: lighter lines for SOC boundaries within same SOH
for i in soc_change_idx:
    plt.axhline(i - 0.5, linewidth=0.5)
    plt.axvline(i - 0.5, linewidth=0.5)

plt.tight_layout()
plt.show()


In [ ]:
import re
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load your 65x65 distance matrix (index and columns are the labels)
df_dist = pd.read_csv(os.path.join(ARTIFACT_DIR, "sample_pairwise_distances_euclidean.csv"), index_col=0)

# --------- Parse label: G{g}-CELLNAME-SOH:{soh}-SOC:{soc} ----------
pat = re.compile(
    r"^(?P<G>G\d+)-(?P<CELL>.+?)-SOH:(?P<SOH>[+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)-SOC:(?P<SOC>[+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)$"
)

def parse_label(label: str):
    m = pat.match(label)
    if not m:
        return {"G": None, "CELL": None, "SOH": np.nan, "SOC": np.nan}
    d = m.groupdict()
    d["SOH"] = float(d["SOH"])
    d["SOC"] = float(d["SOC"])
    return d

meta = df_dist.index.to_series().apply(parse_label).apply(pd.Series)
meta.index = df_dist.index

# --------- Make output directory (optional) ----------
out_dir = f"{ARTIFACT_DIR}cell_heatmaps"
os.makedirs(out_dir, exist_ok=True)

# --------- Plot one heatmap per CELL ----------
cells = sorted(meta["CELL"].dropna().unique())

for cell in cells:
    idx = meta.index[meta["CELL"] == cell]
    if len(idx) < 2:
        continue

    order = meta.loc[idx].sort_values(["SOH", "SOC"], ascending=[False, False]).index
    M = df_dist.loc[order, order].round(2)

    n = len(order)
    plt.figure(figsize=(max(6, n*0.5), max(5, n*0.5)))
    plt.imshow(M.values)
    plt.colorbar(label="Euclidean distance")
    plt.title(f"{cell}: Pairwise distances\n(SOH↓, SOC↓)")
    plt.xlabel("Samples")
    plt.ylabel("Samples")

    plt.xticks(range(n), order, rotation=90, fontsize=7)
    plt.yticks(range(n), order, fontsize=7)

    # Add distance values inside each block
    for i in range(n):
        for j in range(n):
            plt.text(
                j, i, f"{M.iat[i, j]:.2f}",
                ha="center", va="center",
                fontsize=7
            )

    plt.tight_layout()
    out_path = os.path.join(out_dir, f"heatmap_{cell}_with_values.png")
    plt.savefig(out_path, dpi=200)
    plt.close()

print(f"Saved {len(cells)} cell heatmaps to: {out_dir}/")


Method2:

In [ ]:
estimator, out_df, artifact = offline_train(
    offline_csv="../fulldf_date_removeAbOod_G40L80SOC_best_temp25.csv",
    feature_cols=['R0','R1','R2','R3','C1','n1','C2','n2', 'C3','n3','Aw'],
    id_cols=['CELL','SOH_label','SOC_label'],
    clustering_method="boleol",
    k_init_bins=8,
    gamma_merge=0.6,
    soh_estimator_mode="mean",
    k_final=None,
)
